# 训练 decoder：三组 loss 对比（CartPole 主线）

对应脚本：`train_decoder.py`，config 在 `config/train_decoder/<环境>/<weight_mode>.json`。

三组实验唯一区别 = loss 里权重 w 的来源（详见 `report/2026-07-06.md` 第 4 节）：
- `intensity` -> 论文 baseline（暗像素规则，Eq.7）
- `saliency`  -> 我们的方法（w = 1 + α·H，H 为 occlusion 热图）
- `hybrid`    -> 诊断组（已否决：intensity 会稀释 saliency）

**运行前**：kernel 切换成 `starv_shared`（`/home/tealab_shared/starv/env/starv_shared`）。
本文件放在 `verifiable_wm/notebooks/` 下，第一个 cell 会自动定位仓库根目录。

In [ ]:
import os
os.environ.setdefault("PYGLET_HEADLESS", "True")

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "dynamic.py").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "dynamic.py").exists(), (
    f"没找到仓库根目录（dynamic.py），请从 verifiable_wm/ 或 verifiable_wm/notebooks/ 启动 Jupyter"
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

In [ ]:
from utils import load_config, set_seed, resolve_device
import train_decoder as td

def run_train(env_name, weight_mode, alpha=None, seed=None, output_dir=None):
    """跑一组 decoder 训练。
    weight_mode: intensity / saliency / hybrid
    alpha / seed: 不填用 config 默认值；填了则覆盖，且输出目录自动加后缀避免覆盖主结果
    """
    config_path = REPO_ROOT / "config" / "train_decoder" / env_name / f"{weight_mode}.json"
    config = load_config(config_path)
    if alpha is not None:
        config["weight"]["alpha"] = alpha
    if seed is not None:
        config["training"]["seed"] = seed
    if output_dir is not None:
        config["output_dir"] = str(output_dir)
    else:
        suffix = ""
        if alpha is not None:
            suffix += f"_alpha{alpha:g}"
        if seed is not None:
            suffix += f"_seed{seed}"
        config["output_dir"] += suffix
    print(f"[Config] {config_path}")
    print(f"[Output] {config['output_dir']}")
    set_seed(config["training"]["seed"])
    device = resolve_device(config.get("device", "auto"))
    td.train(config, device)

## Case 1: baseline（intensity，论文复现）

In [ ]:
run_train("cartpole", "intensity")

## Case 2: saliency（我们的方法）

In [ ]:
run_train("cartpole", "saliency")

## Case 3: hybrid（诊断组，已否决，一般不用重跑）

In [ ]:
# run_train("cartpole", "hybrid")

## α 扫描（下一步计划）

在 val controller MSE 上选最优 α；α=4 的结果已在主目录，无需重跑。

In [ ]:
# for alpha in [1, 2, 8, 16]:
#     run_train("cartpole", "saliency", alpha=alpha)

## 汇总：读取各实验的 test 指标

In [ ]:
import json
from pathlib import Path

print(f"{'实验':<24}{'best@':>6}{'ctrl_mse':>12}{'pixel_mse':>12}")
for mfile in sorted(Path("dwm_weight/now_weight/cartpole").glob("*/metrics.json")):
    m = json.load(open(mfile))
    t = m["test"]
    print(f"{mfile.parent.name:<24}{m['best_epoch']:>6}{t['ctrl_mse']:>12.5f}{t['pixel_mse']:>12.5f}")

## 闭环 rollout：生成 WM 轨迹（每个变体约 2 分钟，已生成过可跳过）

原理：用 decoder 顶替 camera 跑 30 步闭环（`sampling.py` 的 `rollout_dwm_trajectory`），
与真实轨迹（`real_trajectories.npz`，`make_decoder_dataset.py` 的 `rollout_real_trajectory` 生成）
同起点逐步对比。等价命令行：`python sampling.py config/sampling/cartpole.json --decoder-variant saliency`

In [ ]:
import sampling as sp

def run_sampling_variant(env_name, variant):
    config = load_config(REPO_ROOT / "config" / "sampling" / f"{env_name}.json")
    config["decoder"]["variant"] = variant
    sp.generate_dataset(config)

# run_sampling_variant("cartpole", "intensity")
# run_sampling_variant("cartpole", "saliency")

## 闭环 rollout：轨迹偏差对比（决定性指标②）

每对轨迹逐步算全状态 L2 偏差 d_t = ||s_t_wm − s_t_real||；
「轨迹最大偏差」即论文 conformal inflation（Eq.23）用的量，p95 直接预演 Δ_0.95 的松紧。

In [ ]:
import numpy as np

root = Path("datasets/cartpole/data/dataset_v1")
real = np.load(root / "real_trajectories.npz")

print(f"{'变体':<12}{'平均每步':>10}{'末步':>10}{'最大偏差均值':>14}{'最大偏差p95':>13}")
for variant in ["old", "intensity", "saliency"]:
    f = root / f"dwm_trajectories_{variant}.npz"
    if not f.exists():
        print(f"{variant:<12} (未生成)")
        continue
    dwm = np.load(f)
    r, w = real["test_traj"], dwm["test_traj"]
    assert (r[:, 0] == w[:, 0]).all(), "起点不一致，无法配对对比"
    d = np.linalg.norm(w - r, axis=2)          # (N, 31) 每步偏差
    dmax = d.max(axis=1)                       # 每条轨迹的最大偏差
    print(f"{variant:<12}{d.mean():>10.4f}{d[:, -1].mean():>10.4f}"
          f"{dmax.mean():>14.4f}{np.percentile(dmax, 95):>13.4f}")